### CashFlow Mock

Generates synthetic **CashFlow** (\~3,150 rows) and **CashFlowForecast** (\~181 rows) DataFrames
from a BillingDocument source, with DSO modelling based on customer tier and dunning level.

| Target table | Rows |
|---|---|
| `h2b_dbx_cashflow.cashflow.cashflow` | ~3,150 |
| `h2b_dbx_cashflow.cashflow.cashflow` | ~181 |

**Seed:** 42 &nbsp;|&nbsp; **Forecast window:** 2026-01-01 → 2026-06-30

In [0]:
import numpy as np
import pandas as pd
from datetime import timedelta, datetime, timezone
from decimal import Decimal
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DecimalType
)

# ---------------------------------------------------------------------------
# Tier map (hardcoded — do not rely on CustomerABCClassification)
# ---------------------------------------------------------------------------
_CUSTOMERS = [
    '10100006', '10100002', '12200001',
    '10100012', '10186001', '10186002', '10186003',
    'EWM10-CU01', 'EWM10-CU02', 'EWM10-CU03',
]

_TIER_MAP = {
    '10100006': 'A', '10100002': 'A', '12200001': 'A',
    '10100012': 'B', '10186001': 'B', '10186002': 'B', '10186003': 'B',
    'EWM10-CU01': 'C', 'EWM10-CU02': 'C', 'EWM10-CU03': 'C',
}

# DSO parameters by tier: (mean, std)
_DSO_PARAMS   = {'A': (30, 5), 'B': (45, 8), 'C': (65, 20)}

# Dunning adjustments (days)
_DUNNING_ADJ  = {0: 0, 1: 10, 2: 25, 3: 50}

# Base collection-rate ranges by tier
_BASE_RATE    = {'A': (0.96, 0.99), 'B': (0.92, 0.97), 'C': (0.80, 0.92)}

# Dunning multipliers
_DUNNING_MULT = {0: 1.00, 1: 0.97, 2: 0.93, 3: 0.85}

In [0]:
# ---------------------------------------------------------------------------
# Schemas
# ---------------------------------------------------------------------------
_CF_SCHEMA = StructType([
    StructField("CashFlowID",                   StringType(),       False),
    StructField("CshFlwValdtyStrtDteTmeVal",     DecimalType(21, 7), False),
    StructField("CompanyCode",                   StringType()),
    StructField("TransactionDate",               DateType()),
    StructField("PostingDate",                   DateType()),
    StructField("TransactionCurrency",           StringType()),
    StructField("AmountInTransactionCurrency",   DecimalType(34, 4)),
    StructField("CompanyCodeCurrency",           StringType()),
    StructField("AmountInCompanyCodeCurrency",   DecimalType(34, 4)),
    StructField("GlobalCurrency",                StringType()),
    StructField("AmountInGlobalCurrency",        DecimalType(34, 4)),
    StructField("BankAccountInternalID",         StringType()),
])

_FCF_SCHEMA = StructType([
    StructField("CashFlowID",                   StringType(),       False),
    StructField("CshFlwValdtyStrtDteTmeVal",     DecimalType(21, 7), False),
    StructField("CompanyCode",                   StringType()),
    StructField("TransactionDate",               DateType()),
    StructField("PostingDate",                   DateType()),
    StructField("TransactionCurrency",           StringType()),
    StructField("AmountInTransactionCurrency",   DecimalType(34, 4)),
    StructField("CompanyCodeCurrency",           StringType()),
    StructField("AmountInCompanyCodeCurrency",   DecimalType(34, 4)),
    StructField("GlobalCurrency",                StringType()),
    StructField("AmountInGlobalCurrency",        DecimalType(34, 4)),
])


def _validity_ts(d) -> float:
    """int(PostingDate.timestamp()) encoded as Decimal(21,7)."""
    dt_utc = datetime.combine(d, datetime.min.time(), tzinfo=timezone.utc)
    return float(int(dt_utc.timestamp()))


# ---------------------------------------------------------------------------
# Main generator
# ---------------------------------------------------------------------------
def generate_cashflow_tables(
    billing_df: DataFrame,
    forecast_months: int = 6,
) -> dict:
    """
    Generate CashFlow and CashFlowForecast DataFrames.

    Parameters
    ----------
    billing_df       : Spark DataFrame  – BillingDocument source
    forecast_months  : int              – months to forecast (default 6)

    Returns
    -------
    dict  {'CashFlow': spark_df, 'CashFlowForecast': spark_df}
    """
    spark = SparkSession.builder.getOrCreate()
    rng   = np.random.default_rng(seed=42)

    # ── 1. Read customer dunning ────────────────────────────────────────────
    customer_dunning_df = (
        spark.table("h2b_bdc_customer.customer.customerdunning")
        .filter(F.col("Customer").isin(_CUSTOMERS))
        .select("Customer", "DunningLevel")
    )
    dunning_map = {
        row["Customer"]: int(row["DunningLevel"]) if row["DunningLevel"] else 0
        for row in customer_dunning_df.collect()
    }

    # ── 2. Filter billing documents ─────────────────────────────────────────
    filtered = (
        billing_df
        .filter(
            (F.col("BillingDocumentIsCancelled") == False)
            & (F.col("BillingDocumentType") == "F2")
        )
        .select(
            "PayerParty", "BillingDocumentDate",
            "TotalNetAmount", "TransactionCurrency",
        )
    )
    rows_pdf = filtered.toPandas()

    # ── 3. Build CashFlow rows ──────────────────────────────────────────────
    cf_rows = []
    for _, row in rows_pdf.iterrows():
        payer = row["PayerParty"]
        tier  = _TIER_MAP.get(payer)
        if tier is None:
            continue

        billing_date = pd.Timestamp(row["BillingDocumentDate"])
        total_net    = float(row["TotalNetAmount"])
        currency     = row["TransactionCurrency"]

        # DSO -----------------------------------------------------------
        dunning_level = dunning_map.get(payer, 0)
        mu, sigma     = _DSO_PARAMS[tier]
        base_dso      = max(1, round(float(rng.normal(mu, sigma))))
        dunning_adj   = _DUNNING_ADJ.get(dunning_level, 0)
        noise         = round(float(rng.normal(0, 3)))
        total_dso     = max(1, min(base_dso + dunning_adj + noise, 180))

        posting_date = (billing_date + timedelta(days=int(total_dso))).date()

        # Amount --------------------------------------------------------
        lo, hi    = _BASE_RATE[tier]
        base_rate = float(rng.uniform(lo, hi))
        d_mult    = _DUNNING_MULT.get(dunning_level, 1.0)
        amount    = round(total_net * base_rate * d_mult, 4)

        # Currency conversion -------------------------------------------
        amt_cc  = amount if currency == "EUR" else round(amount * 0.92, 4)
        bank_id = "BANK001" if currency == "EUR" else "BANK002"

        cf_rows.append({
            "CashFlowID":                   f"CF-{len(cf_rows)+1:06d}",
            "CshFlwValdtyStrtDteTmeVal":     _validity_ts(posting_date),
            "CompanyCode":                   "CC01",
            "TransactionDate":               posting_date,
            "PostingDate":                   posting_date,
            "TransactionCurrency":           currency,
            "AmountInTransactionCurrency":   amount,
            "CompanyCodeCurrency":           "EUR",
            "AmountInCompanyCodeCurrency":   amt_cc,
            "GlobalCurrency":                "EUR",
            "AmountInGlobalCurrency":        amt_cc,
            "BankAccountInternalID":         bank_id,
        })

    cf_pdf      = pd.DataFrame(cf_rows)

    # Convert float columns to Decimal for Arrow/Spark DecimalType compatibility
    _dec_cols = ["CshFlwValdtyStrtDteTmeVal", "AmountInTransactionCurrency",
                 "AmountInCompanyCodeCurrency", "AmountInGlobalCurrency"]
    for c in _dec_cols:
        cf_pdf[c] = cf_pdf[c].apply(lambda x: Decimal(str(x)))

    cashflow_df = spark.createDataFrame(cf_pdf, schema=_CF_SCHEMA)

    # ── 4. Build CashFlowForecast ───────────────────────────────────────────
    # Step 1: weekly average of AmountInCompanyCodeCurrency over 2023-2025
    cf_pdf["_pd"]       = pd.to_datetime(cf_pdf["PostingDate"])
    hist                = cf_pdf[(cf_pdf["_pd"].dt.year >= 2023) & (cf_pdf["_pd"].dt.year <= 2025)].copy()
    hist["iso_week"]    = hist["_pd"].dt.isocalendar().week.values
    hist["_amt"]        = hist["AmountInCompanyCodeCurrency"].astype(float)
    weekly_avg          = hist.groupby("iso_week")["_amt"].mean()

    # Step 2: overall mean of weekly averages
    overall_mean = float(weekly_avg.mean()) if len(weekly_avg) > 0 else 0.0

    # Step 3: daily forecast 2026-01-01 → 2026-01-01 + forecast_months
    fc_start = pd.Timestamp("2026-01-01")
    fc_end   = fc_start + pd.DateOffset(months=forecast_months) - timedelta(days=1)
    fc_dates = pd.date_range(fc_start, fc_end, freq="D")

    fcf_rows = []
    for d in fc_dates:
        iso_wk = d.isocalendar().week
        si     = (weekly_avg.get(iso_wk, overall_mean) / overall_mean) if overall_mean else 1.0
        forecast    = round((overall_mean / 7) * si * 1.04 * float(rng.uniform(0.92, 1.08)), 4)

        fcf_rows.append({
            "CashFlowID":                   f"FCF-{len(fcf_rows)+1:06d}",
            "CshFlwValdtyStrtDteTmeVal":     _validity_ts(d.date()),
            "CompanyCode":                   "CC01",
            "TransactionDate":               d.date(),
            "PostingDate":                   d.date(),
            "TransactionCurrency":           "EUR",
            "AmountInTransactionCurrency":   forecast,
            "CompanyCodeCurrency":           "EUR",
            "AmountInCompanyCodeCurrency":   forecast,
            "GlobalCurrency":                "EUR",
            "AmountInGlobalCurrency":        forecast,
        })

    fcf_pdf     = pd.DataFrame(fcf_rows)

    # Convert float columns to Decimal for Arrow/Spark DecimalType compatibility
    _fcf_dec_cols = ["CshFlwValdtyStrtDteTmeVal", "AmountInTransactionCurrency",
                     "AmountInCompanyCodeCurrency", "AmountInGlobalCurrency"]
    for c in _fcf_dec_cols:
        fcf_pdf[c] = fcf_pdf[c].apply(lambda x: Decimal(str(x)))

    forecast_df = spark.createDataFrame(fcf_pdf, schema=_FCF_SCHEMA)

    return {"CashFlow": cashflow_df, "CashFlowForecast": forecast_df}

In [0]:
# ---------------------------------------------------------------------------
# Read source & generate
# ---------------------------------------------------------------------------
billing_df = spark.table("h2b_dbx_billingdocument.billingdocument.billingdocument")

result = generate_cashflow_tables(billing_df, forecast_months=6)

cashflow_df  = result["CashFlow"]
forecast_df  = result["CashFlowForecast"]

print(f"CashFlow rows:         {cashflow_df.count():,}")
print(f"CashFlowForecast rows: {forecast_df.count():,}")

display(cashflow_df)
display(forecast_df)

In [0]:
# ---------------------------------------------------------------------------
# Write to Unity Catalog
# ---------------------------------------------------------------------------
spark.sql("CREATE CATALOG IF NOT EXISTS h2b_dbx_cashflow")
spark.sql("CREATE SCHEMA IF NOT EXISTS h2b_dbx_cashflow.cashflow")

cashflow_df.write.mode("overwrite").saveAsTable("h2b_dbx_cashflow.cashflow.cashflow")
forecast_df.write.mode("overwrite").saveAsTable("h2b_dbx_cashflow.cashflow.cashflowforecast")

print("\n✓ Written to h2b_dbx_cashflow.cashflow.cashflow")
print("✓ Written to h2b_dbx_cashflow.cashflow.cashflowforecast")

In [0]:
spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflow ALTER COLUMN CashFlowID SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflow ALTER COLUMN CshFlwValdtyStrtDteTmeVal SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflow ADD CONSTRAINT cashflow_pk PRIMARY KEY(CashFlowID, CshFlwValdtyStrtDteTmeVal)")

spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflowforecast ALTER COLUMN CashFlowID SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflowforecast ALTER COLUMN CshFlwValdtyStrtDteTmeVal SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_cashflow.cashflow.cashflowforecast ADD CONSTRAINT cashflowforecast_pk PRIMARY KEY(CashFlowID, CshFlwValdtyStrtDteTmeVal)")